In [0]:
from pyspark import pipelines as dp
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.functions import col, from_json, to_timestamp, current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    BooleanType,
)

# Schema for the JSON payload in the bronze value column
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("card_number", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("merchant_name", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("payment_channel", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
    StructField("is_international", BooleanType(), True),
    StructField("status", StringType(), True),
])

@dp.expect_or_drop("valid_transaction_id", "transaction_id IS NOT NULL")
@dp.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")
@dp.expect_or_drop("valid_card_number", "card_number IS NOT NULL")
@dp.expect_or_drop("valid_merchant_id", "merchant_id IS NOT NULL")
@dp.expect_or_drop("valid_merchant_name", "merchant_name IS NOT NULL")
@dp.expect_or_drop("valid_merchant_category", "merchant_category IS NOT NULL")
@dp.expect("valid_amount", "amount > 0")


@dp.table(
    name="finguard.silver.transaction",
    comment="Silver streaming table with parsed transaction events from bronze.",
)
def transaction_silver() -> DataFrame:
    bronze = spark.readStream.table("finguard.bronze.transaction")

    parsed = bronze.select(
        from_json(col("value"), transaction_schema).alias("payload"),
        col("key"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("kafka_timestamp"),
        col("timestampType"),
        col("_ingested_at").alias("bronze_ingested_at"),
    )

    return parsed.select(
        col("payload.transaction_id").alias("transaction_id"),
        col("payload.customer_id").alias("customer_id"),
        col("payload.card_number").alias("card_number"),
        col("payload.merchant_id").alias("merchant_id"),
        col("payload.merchant_name").alias("merchant_name"),
        col("payload.merchant_category").alias("merchant_category"),
        col("payload.amount").alias("amount"),
        col("payload.currency").alias("currency"),
        col("payload.transaction_type").alias("transaction_type"),
        col("payload.payment_channel").alias("payment_channel"),
        col("payload.device_id").alias("device_id"),
        col("payload.city").alias("city"),
        col("payload.country").alias("country"),
        to_timestamp(col("payload.transaction_timestamp")).alias("transaction_timestamp"),
        col("payload.is_international").alias("is_international"),
        col("payload.status").alias("status"),
        col("key"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("kafka_timestamp"),
        col("timestampType"),
        col("bronze_ingested_at"),
        current_timestamp().alias("silver_ingested_at"),
    )